### AST

AST is pretrained on AudioSet (5800 hours, 527 audio classes), so it already understands audio really well. We just finetune it on our genre task

### Imports

In [17]:
import os
import random
import glob
import warnings
import time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
import torchaudio
import librosa
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")

SEED = 23456543
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

Using: cuda


### Config

In [4]:
DATA_ROOT = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
OUTPUT_DIR = "./outputs"

STEMS_DIR = os.path.join(DATA_ROOT, "genres_stems")
NOISE_DIR = os.path.join(DATA_ROOT, "ESC-50-master", "audio")
TEST_DIR = os.path.join(DATA_ROOT, "mashups")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")

# AST expects 16kHz audio
SR = 16000
DURATION = 10.0
TARGET_LEN = int(SR * DURATION)

# genres
GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                 'jazz', 'metal', 'pop', 'reggae', 'rock'])
GENRES2IDX = {g : i for i, g in enumerate(GENRES)}
IDX2GENRES = {i : g for g, i in GENRES2IDX.items()}
STEMS = ["drums", "vocals", "bass", "other"]

# training params - less than CNNs cuz this is very heavy (transformer)
SAMPLES_PER_GENRE = 800 # 8000 mashups per epoch
BATCH_SIZE = 8
ACCUM_STEPS = 4 # effective batch = 8 * 4 = 32
EPOCHS = 20
LR_BACKBONE = 1e-5 # lower to preserve pretrained weights
LR_HEAD = 1e-3 # high for new classifier head
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
NUM_WORKERS = 4
WARMUP_EPOCHS = 2

STEM_WEIGHTS = {
    "drums": 0.45,
    "vocals": 0.30,
    "bass": 0.15,
    "other": 0.10
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Will train on {SAMPLES_PER_GENRE * 10} mashups/epoch")
print(f"Effective batch size: {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE * ACCUM_STEPS}")

Will train on 8000 mashups/epoch
Effective batch size: 8 x 4 = 32


### Index & Split

In [5]:
stem_index = {g : {st : [] for st in STEMS} for g in GENRES}
song_index = {g : [] for g in GENRES}

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)
    songs = sorted(s for s in os.listdir(genre_path)
                  if os.path.isdir(os.path.join(genre_path, s)))

    for song in songs:
        song_dir = os.path.join(genre_path, song)
        available_stems = []
        for stem in STEMS:
            filepath = os.path.join(song_dir, f"{stem}.wav")
            if os.path.exists(filepath):
                stem_index[genre][stem].append(filepath)
                available_stems.append(stem)

        if available_stems:
            song_index[genre].append({
                'dir': song_dir,
                'stems': available_stems
            })

noise_files = sorted(glob.glob(os.path.join(NOISE_DIR, "*.wav")))
print(f"Found {len(noise_files)} noise clips from ESC-50")

Found 2000 noise clips from ESC-50


In [6]:
train_stems = {g : {st: [] for st in STEMS} for g in GENRES}
val_songs = {g: [] for g in GENRES}

for genre in GENRES:
    songs = song_index[genre].copy()
    random.shuffle(songs)
    split_point = int(0.85 * len(songs))

    # slice
    train_list = songs[:split_point]
    val_list = songs[split_point:]
    val_songs[genre] = val_list

    # only use stems from training songs
    train_dirs = {s['dir'] for s in train_list}
    for stem in STEMS:
        train_stems[genre][stem] = [
            fp for fp in stem_index[genre][stem]
            if os.path.dirname(fp) in train_dirs
        ]

    print(f"{genre}: {len(train_list)} train, {len(val_list)} val")

blues: 85 train, 15 val
classical: 85 train, 15 val
country: 85 train, 15 val
disco: 85 train, 15 val
hiphop: 85 train, 15 val
jazz: 85 train, 15 val
metal: 85 train, 15 val
pop: 85 train, 15 val
reggae: 85 train, 15 val
rock: 85 train, 15 val


In [ ]:
# caching

CACHE_DIR = "./wav_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(orig_path):
    # flatten path into a safe filename
    key = orig_path.replace("/", "_")
    return os.path.join(CACHE_DIR, key + ".npy")

def precompute_cache(filepaths):
    for fp in tqdm(filepaths, desc="Caching waveforms"):
        cp = cache_path(fp)
        if os.path.exists(cp):
            continue
        try:
            y, _ = librosa.load(fp, sr=SR, mono=True)
        except Exception:
            y = np.zeros(TARGET_LEN, dtype=np.float32)
        np.save(cp, y.astype(np.float32))

# gather every file that will ever be loaded
all_files = set()
for genre in GENRES:
    for stem in STEMS:
        all_files.update(stem_index[genre][stem])
all_files.update(noise_files)

precompute_cache(list(all_files))

### Helper function

NOte: AST uses 16kHz. The AST Feature extractor handles mel conversion internally, so we just need to load raw waveforms.

In [8]:
def load_wav(path, sr=SR, target_len=TARGET_LEN):
    cp = cache_path(path)
    try:
        y = np.load(cp)
    except Exception:
        y = np.zeros(target_len, dtype=np.float32)
    # clip or pad happens per-call since crop is random each time
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        start = random.randint(0, len(y) - target_len)
        y = y[start:start + target_len]
    return y.astype(np.float32)

def load_wav_tta(path, sr=SR, target_len=TARGET_LEN):
    cp = cache_path(path)
    try:
        y = np.load(cp)
    except Exception:
        try:
            y, _ = librosa.load(path, sr=sr, mono=True)
        except Exception:
            y = np.zeros(target_len, dtype=np.float32)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        start = random.randint(0, len(y) - target_len)  # random, not center — this is what makes TTA rounds differ
        y = y[start:start + target_len]
    return y.astype(np.float32)

### Datasets

In [9]:
class MashupDataset(Dataset):
    def __init__(self, stem_idx, noise_files, samples_per_genre=800, augment=True):
        self.stem_idx = stem_idx
        self.noise_files = noise_files
        self.augment = augment
        self.samples = []
        for genre in GENRES:
            for _ in range(samples_per_genre):
                self.samples.append(GENRES2IDX[genre])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        genre_idx = self.samples[idx]
        genre = IDX2GENRES[genre_idx]

        # pick random stems from different songs of same genre
        stems_wav = []
        for stem_type in STEMS:
            available = self.stem_idx[genre][stem_type]
            if not available:
                continue
            wav = load_wav(random.choice(available))
            # weight by stem importance
            gain = random.uniform(0.5, 1.5) * (STEM_WEIGHTS[stem_type] / 0.25)
            stems_wav.append(wav * gain)

        if not stems_wav:
            mel = np.zeros(TARGET_LEN, dtype=np.float32)
            return torch.from_numpy(mel).unsqueeze(0), genre_idx

        # mix all stems
        mix = np.sum(stems_wav, axis=0)

        if self.augment:
            # circular time shift
            mix = np.roll(mix, random.randint(-SR, SR))

            # add ESC-50 noise
            for _ in range(random.randint(0, 2)):
                noise = load_wav(random.choice(self.noise_files))
                snr_db = random.uniform(5.0, 25.0)
                sig_pwr = np.mean(mix ** 2) + 1e-10
                nse_pwr = np.mean(noise ** 2) + 1e-10
                scale = np.sqrt(sig_pwr / (nse_pwr * 10 ** (snr_db / 10)))
                mix = mix + noise * scale

            # overdrive (30% chance)
            if random.random() < 0.3:
                mix = np.clip(mix * random.uniform(1.2, 3.0), -1, 1)

        # normalize
        peak = np.max(np.abs(mix))
        if peak > 1e-6:
            mix = mix / peak * random.uniform(0.7, 1.0)
        
        return torch.from_numpy(mix).float(), genre_idx

In [10]:
class ValDataset(Dataset):
    def __init__(self, song_index):
        self.items = []
        for genre in GENRES:
            for song in song_index[genre]:
                self.items.append((song, GENRES2IDX[genre]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        song_info, label = self.items[idx]
        stems = [load_wav(os.path.join(song_info['dir'], f"{st}.wav"))
                for st in song_info['stems']]
        mix = np.sum(stems, axis=0)
        peak = np.max(np.abs(mix))
        if peak > 1e-6:
            mix = mix / peak
        return torch.from_numpy(mix).float(), label

class TestDataset(Dataset):
    def __init__(self, test_dir, test_csv):
        self.df = pd.read_csv(test_csv, dtype={'id': str})
        self.paths = []
        for _, row in self.df.iterrows():
            path = None
            for pattern in [f"song{str(row['id']).zfill(4)}.wav",
                          f"{row['id']}.wav",
                          f"song{row['id']}.wav"]:
                p = os.path.join(test_dir, pattern)
                if os.path.exists(p):
                    path = p
                    break
            self.paths.append(path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.paths[idx]
        if path:
            wav = load_wav_tta(path)
            peak = np.max(np.abs(wav))
            if peak > 1e-6:
                wav = wav / peak
            wav_tensor = torch.from_numpy(wav).float()
        else:
            wav_tensor = torch.zeros(TARGET_LEN, dtype=torch.float32)
        return wav_tensor, str(self.df.iloc[idx]['id'])

### Model

Audio Spectrogram Transformer(AST) Model from MIT, pretrained on AudioSet (527 classes). Replace the head with 10 class head for our PS

The feature extractor converts raw waveforms into the format AST expects. We use collator functions to do this in the dataloader

In [11]:
AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)
print("Feature extractor loaded")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Feature extractor loaded


In [12]:
class ASTGenreClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.ast = ASTForAudioClassification.from_pretrained(
            AST_MODEL_NAME,
            num_labels=num_classes,
            ignore_mismatched_sizes = True # head size changes from 527 -> 10
        )

    def forward(self, x):
        # input_values shape: (batch, 1024, 128) - from feature extractor
        outputs = self.ast(input_values=x)
        return outputs.logits

model = ASTGenreClassifier(num_classes=10).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"AST parameters: {total_params / 1e6:.1f}M")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


AST parameters: 86.2M


### Collator Functions

The AST feature extractor needs to run on batches of raw waveforms. We use custom collator functions to do this inside the DataLoader.

In [13]:
class ASTCollator:
    def __init__(self, feature_extractor, sr=16000):
        self.fe = feature_extractor
        self.sr = sr

    def __call__(self, batch):
        waveforms, labels = zip(*batch)

        waveforms_np = [w.numpy() for w in waveforms]
        inputs = self.fe(
            waveforms_np,
            sampling_rate=self.sr,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True
        )

        if isinstance(labels[0], int) or isinstance(labels[0], np.integer):
            labels_out = torch.tensor(labels, dtype=torch.long)
        else:
            labels_out = labels

        return inputs["input_values"], labels_out

class ASTTestCollator:
    # same but for test set where labels are string IDs
    
    def __init__(self, feature_extractor, sr=16000):
        self.fe = feature_extractor
        self.sr = sr

    def __call__(self, batch):
        waveforms, ids = zip(*batch)
        waveforms_np = [w.numpy() for w in waveforms]
        inputs = self.fe(
            waveforms_np,
            sampling_rate=self.sr,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )
        return inputs["input_values"], list(ids)

print("Collators ready")

Collators ready


### Training & Eval

In [14]:
def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0.0
    num_samples = 0
    optimizer.zero_grad()

    for step, (input_values, labels) in enumerate(tqdm(loader, desc="Train", leave=False)):
        input_values = input_values.to(DEVICE)
        labels = labels.to(DEVICE)

        with autocast():
            logits = model(input_values)
            loss = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() *  ACCUM_STEPS * input_values.size(0)
        num_samples += len(labels)

    return total_loss / num_samples

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for input_values, labels in loader:
        input_values = input_values.to(DEVICE)
        with autocast():
            logits = model(input_values)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

    f1 = f1_score(all_labels, all_preds, average="macro")
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return f1, acc, np.array(all_preds), np.array(all_labels)

### Optimizer with Differential Learning Rate

This is important - the AST backbone is already pretrained on AudioSet, so we use a very low LR (1e-5) to not destroy those learned representations. The new classification head gets a higher LR (1e-3) since it needs to learn from scratch.

We also use a cosine schedule with 2 epochs of warmup to let the head stabilize before the backbone starts changing.

In [15]:
backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

print(f"Backbone: {sum(p.numel() for p in backbone_params) / 1e6:.1f}M params (lr={LR_BACKBONE})")
print(f"Head: {sum(p.numel() for p in head_params)} params (lr={LR_HEAD})")

# different learning rates for backbone vs head
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# cosine schedule with warmup
def get_lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        # linear warmup
        return (epoch + 1) / WARMUP_EPOCHS
    # cosine decay after warmup
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = GradScaler()

Backbone: 86.2M params (lr=1e-05)
Head: 9226 params (lr=0.001)


In [ ]:
collator = ASTCollator(feature_extractor, sr=SR)

val_ds = ValDataset(val_songs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        collate_fn=collator)
print(f"Validation samples: {len(val_ds)}")

best_f1 = 0.0
history = {'loss': [], 'val_f1': [], 'val_acc': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # new mashups every epoch
    train_ds = MashupDataset(train_stems, noise_files, SAMPLES_PER_GENRE, augment=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              drop_last=True, collate_fn=collator)

    loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
    scheduler.step()
    
    val_f1, val_acc, _, _ = evaluate(model, val_loader)
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - start_time

    # track history
    history['loss'].append(loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)

    # save best
    tag = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_ast.pth'))
        tag = " [best]"

    print(f"E{epoch:02d}/{EPOCHS} | loss={loss:.4f} | f1={val_f1:.4f} | "
          f"acc={val_acc:.4f} | lr={lr:.6f} | {elapsed:.0f}s{tag}")

print(f"\nBest validation F1: {best_f1:.4f}")

Validation samples: 150 \
E01/20 | loss=0.8812 | f1=0.8510 | acc=0.8533 | lr=0.000010 | 559s [best] \
E02/20 | loss=0.6884 | f1=0.8470 | acc=0.8467 | lr=0.000010 | 572s \
E03/20 | loss=0.6265 | f1=0.8531 | acc=0.8533 | lr=0.000010 | 570s [best] \
E04/20 | loss=0.6040 | f1=0.8422 | acc=0.8400 | lr=0.000010 | 570s \
E05/20 | loss=0.5781 | f1=0.8690 | acc=0.8667 | lr=0.000009 | 569s [best] \
E06/20 | loss=0.5640 | f1=0.8911 | acc=0.8933 | lr=0.000009 | 569s [best] \
E07/20 | loss=0.5606 | f1=0.8803 | acc=0.8800 | lr=0.000008 | 568s \
E08/20 | loss=0.5480 | f1=0.8393 | acc=0.8400 | lr=0.000008 | 568s \
E09/20 | loss=0.5397 | f1=0.8600 | acc=0.8600 | lr=0.000007 | 568s \
E10/20 | loss=0.5353 | f1=0.8549 | acc=0.8533 | lr=0.000006 | 568s \
E11/20 | loss=0.5283 | f1=0.8500 | acc=0.8467 | lr=0.000005 | 565s \
E12/20 | loss=0.5206 | f1=0.8648 | acc=0.8667 | lr=0.000004 | 564s \
E13/20 | loss=0.5201 | f1=0.8730 | acc=0.8733 | lr=0.000003 | 562s \
E14/20 | loss=0.5146 | f1=0.8739 | acc=0.8733 | lr=0.000003 | 560s \
E15/20 | loss=0.5152 | f1=0.8667 | acc=0.8667 | lr=0.000002 | 559s \
E16/20 | loss=0.5131 | f1=0.8822 | acc=0.8867 | lr=0.000001 | 559s \
E17/20 | loss=0.5133 | f1=0.8860 | acc=0.8867 | lr=0.000001 | 558s \
E18/20 | loss=0.5093 | f1=0.8805 | acc=0.8800 | lr=0.000000 | 558s \
E19/20 | loss=0.5090 | f1=0.8802 | acc=0.8800 | lr=0.000000 | 559s \
E20/20 | loss=0.5089 | f1=0.8786 | acc=0.8800 | lr=0.000000 | 557s 

Best validation F1: 0.8911

### Plots

In [ ]:
# training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_f1'], label='F1')
axes[1].plot(history['val_acc'], label='Accuracy', alpha=0.7)
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

plt.suptitle(f'AST - Best F1: {best_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ast_curves.png'), dpi=150)
plt.show()

![training_curve](../assets/AST/ast_curves.png)

In [ ]:
# confusion matrix
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, '/kaggle/input/models/kaushalvaid/ast-mashup-genre/pytorch/default/1/best_ast.pth'), weights_only=True))
val_f1, val_acc, preds, labels = evaluate(model, val_loader)

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(labels, preds)
ConfusionMatrixDisplay(cm, display_labels=GENRES).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'AST Confusion Matrix - F1={val_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ast_confusion.png'), dpi=150)
plt.show()

print(classification_report(labels, preds, target_names=GENRES))

![confusion_matrix](../assets/AST/ast_confusion.png)

### TTA + Submission

In [18]:
@torch.no_grad()
def predict_tta(model, loader, n_tta=5):
    # run inference multiple times and average
    model.eval()
    
    # collect all IDs first
    all_ids = []
    for _, ids in loader:
        all_ids.extend(ids)

    # run n_tta forward passes
    all_probs = []
    for tta_round in range(n_tta):
        round_probs = []
        for input_values, _ in loader:
            input_values = input_values.to(DEVICE)
            with autocast():
                logits = model(input_values)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            round_probs.append(probs)
        all_probs.append(np.vstack(round_probs))

    # average across rounds
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs.argmax(1), all_ids, avg_probs


# run on test set
test_collator = ASTTestCollator(feature_extractor, sr=SR)
test_ds = TestDataset(TEST_DIR, TEST_CSV)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True,
                         collate_fn=test_collator)
print(f"Test samples: {len(test_ds)}")

preds, ids, probs = predict_tta(model, test_loader, n_tta=5)
print(f"Prediction distribution: {Counter(preds)}")

Test samples: 3020
Prediction distribution: Counter({np.int64(0): 358, np.int64(8): 333, np.int64(9): 331, np.int64(6): 313, np.int64(7): 308, np.int64(5): 308, np.int64(4): 303, np.int64(3): 301, np.int64(2): 252, np.int64(1): 213})


In [19]:
# create submission csv
test_df = pd.read_csv(TEST_CSV, dtype={'id': str})
pred_dict = {str(id_): IDX2GENRES[p] for id_, p in zip(ids, preds)}
test_df['genre'] = test_df['id'].apply(lambda x: pred_dict.get(str(x), 'rock'))
test_df[['id', 'genre']].to_csv(os.path.join(OUTPUT_DIR, 'submission_ast.csv'), index=False)

print("Submission saved!")
print(test_df['genre'].value_counts().sort_index())

# save probabilities for later ensemble with CNN
np.save(os.path.join(OUTPUT_DIR, 'test_probs_ast.npy'), probs)
print("\nProbabilities saved for ensemble use")

Submission saved!
genre
blues        358
classical    213
country      252
disco        301
hiphop       303
jazz         308
metal        313
pop          308
reggae       333
rock         331
Name: count, dtype: int64

Probabilities saved for ensemble use
